In [5]:
import pandas as pd
import numpy as np


train_behaviors_path = "../data/small/train/behaviors.parquet"
train_articles_path = "../data/small/train/articles.parquet"
train_history_path = "../data/small/train/history.parquet"

validation_behaviors_path = "../data/small/validation/behaviors.parquet"
validation_articles_path = "../data/small/validation/articles.parquet"
validation_history_path = "../data/small/validation/history.parquet"

text_emb_path = "../data/roberta_vector.parquet"
image_emb_path = "../data/image_embeddings.parquet"

In [6]:
df_train_behaviors_prob = pd.read_parquet(train_behaviors_path).head(5)
print("="*25 + " behaviors列名及数据类型 " + "="*25)
print(df_train_behaviors_prob.dtypes)


========================= behaviors列名及数据类型 =========================
impression_id                     uint32
article_id                       float64
impression_time           datetime64[us]
read_time                        float32
scroll_percentage                float32
device_type                         int8
article_ids_inview                object
article_ids_clicked               object
user_id                           uint32
is_sso_user                         bool
gender                           float64
postcode                         float64
age                              float64
is_subscriber                       bool
session_id                        uint32
next_read_time                   float32
next_scroll_percentage           float32
dtype: object


In [8]:
display(df_train_behaviors_prob[['impression_id', 'user_id', 'device_type', 'gender', 'age', 'article_ids_inview', 'article_ids_clicked','impression_time']].head(2))

,impression_id,user_id,device_type,gender,age,article_ids_inview,article_ids_clicked,impression_time
0,149474,139836,2,NaN,NaN,"[9778623, 9778682, 9778669, 9778657, 9778736, ...",[9778657],2023-05-24 07:47:53
1,150528,143471,2,NaN,NaN,"[9778718, 9778728, 9778745, 9778669, 9778657, ...",[9778623],2023-05-24 07:33:25


In [14]:
df_train_articles_prob = pd.read_parquet(train_articles_path).head(5)
print("="*25 + " articles列名及数据类型 " + "="*25)
print(df_train_articles_prob.dtypes)

========================= articles列名及数据类型 =========================
article_id                     int32
title                         object
subtitle                      object
last_modified_time    datetime64[us]
premium                         bool
body                          object
published_time        datetime64[us]
image_ids                     object
article_type                  object
url                           object
ner_clusters                  object
entity_groups                 object
topics                        object
category                       int16
subcategory                   object
category_str                  object
total_inviews                float64
total_pageviews              float64
total_read_time              float32
sentiment_score              float32
sentiment_label               object
dtype: object


In [15]:
display(df_train_articles_prob[['article_id', 'article_type', 'topics', 'category', 'subcategory', 'sentiment_label', 'published_time']].head(2))

,article_id,article_type,topics,category,subcategory,sentiment_label,published_time
0,3001353,article_default,"[Kriminalitet, Personfarlig kriminalitet]",140,[],Negative,2006-08-31 08:06:45
1,3003065,article_default,"[Underholdning, Film og tv, Økonomi]",414,"[433, 434]",Positive,2006-05-21 16:57:00


In [20]:
a = {1: 'a', 2: 'b', 3:'c'}

print(list(a.values())[0])
print(type(a.values()))


a
<class 'dict_values'>


In [21]:
df_train_history_prob = pd.read_parquet(train_history_path).head(5)
print("="*25 + " history列名及数据类型 " + "="*25)
print(df_train_history_prob.dtypes)

========================= history列名及数据类型 =========================
user_id                    uint32
impression_time_fixed      object
scroll_percentage_fixed    object
article_id_fixed           object
read_time_fixed            object
dtype: object


In [22]:
display(df_train_history_prob[['user_id', 'article_id_fixed']].head(2))

,user_id,article_id_fixed
0,13538,"[9738663, 9738569, 9738663, 9738490, 9738663, ..."
1,14241,"[9738557, 9738528, 9738533, 9738684, 9739035, ..."


In [7]:
df_text_emb_prob = pd.read_parquet(text_emb_path).head()
print("="*25 + " text列名及数据类型 " + "="*25)
print(df_text_emb_prob.dtypes)

df_image_emb_prob = pd.read_parquet(image_emb_path).head()
print("\n" + "="*25 + " image列名及数据类型 " + "="*25)
print(df_image_emb_prob.dtypes)


========================= text列名及数据类型 =========================
article_id                      int32
FacebookAI/xlm-roberta-base    object
dtype: object

========================= image列名及数据类型 =========================
article_id          int32
image_embedding    object
dtype: object


In [12]:
display(df_text_emb_prob[['article_id', 'FacebookAI/xlm-roberta-base']].head(2))
print(df_text_emb_prob['FacebookAI/xlm-roberta-base'][0].shape)
print("="*50)
display(df_image_emb_prob[['article_id', 'image_embedding']].head(2))
print(df_image_emb_prob['image_embedding'][0].shape)




,article_id,FacebookAI/xlm-roberta-base
0,3000022,"[0.102449246, 0.10114823, 0.056887403, 0.02293..."
1,3000063,"[0.10729711, 0.103072755, 0.054031033, -0.0292..."


(768,)


,article_id,image_embedding
0,9734738,"[-0.023272939, -0.03915197, -0.022547087, 0.04..."
1,8647636,"[-0.04257018, -0.038090054, -0.0064549753, 0.0..."


(1024,)


In [1]:
import torch
from torch.utils.data import Dataset
import polars as pl
import joblib
import gc
import numpy as np
from sklearn.decomposition import PCA

def create_id_mapping(raw_ids_series: pl.Series) -> dict:
    """
    将离散特征映射到[1, 特征数]，0作为padding
    :param raw_ids_series: Polars Series(包含所有唯一的原始ID)
    :return dict: {raw_id: mapped_id}(mapped_id从1开始)
    """
    if raw_ids_series.dtype == pl.List:
        unique_ids = raw_ids_series.explode().drop_nulls().unique().to_list()
    else:
        unique_ids = raw_ids_series.drop_nulls().unique().to_list()
    ids_map = {raw_id:mapped_id + 1 for mapped_id, raw_id in enumerate(unique_ids)}
    return ids_map


def build_offline_article_vault(
        article_path, 
        text_emb_path, 
        image_emb_path, 
        output_dir, 
        emb_dim=64) -> dict:
    """
    将新闻离散特征转化成id；
    保存{mapped_id: {published_time, topics, category, subcategory, sentiment_label, article_type}}字典为。pkl文件；
    保存text_emb,image_emb为npy文件

    :param arcicle_path: 文章数据集路径
    :param text_emb_path: 文本emb数据集路径
    :param image_emb_path: 图片emb数据集路径
    :param output_dir: 输出文件路径
    :param emb_dim: pca降维维度

    :return article_ids_mapping_dict: 文章id映射字典{raw_id:mapped_id}
    """
    print("=" * 50 + " Processing article data " + "=" * 50)
    df_article = (pl.scan_parquet(article_path)
                  .drop_nulls("article_id")
                  .collect())

    article_ids_mapping_dict = create_id_mapping(df_article['article_id'])

    df_article = (
        df_article
        .with_columns(
            pl.col('article_id').replace(article_ids_mapping_dict, default=0),
            pl.col('published_time').dt.timestamp("ms").alias('published_ts'),
            pl.col('subcategory').list.first().fill_null("").alias('subcat'),
        )
        .select(['article_id', 
                'published_ts', 
                'topics', 
                'category', 
                'subcat', 
                'sentiment_label', 
                'article_type'])
    )

    article_discrete_col = ['article_id','article_type','topics','category','subcat','sentiment_label']
    for col in article_discrete_col[1:]:
        col_map = create_id_mapping(df_article[col])
        if df_article[col].dtype == pl.List:
            df_article = df_article.with_columns(
                pl.col(col).list.eval(pl.element().replace(col_map))
            )
        else:
            df_article = df_article.with_columns(pl.col(col).replace(col_map, default=0))
        del col_map

    news_features_dict = {row['article_id']:
                          {'published_ts':row['published_ts'],
                           'article_type':row['article_type'],
                           'topics':row['topics'],
                           'category':row['category'],
                           'subcat':row['subcat'],
                           'sentiment_label':row['sentiment_label']
                           }for row in df_article.iter_rows(named=True)}    # .iter_rows默认返回元组，named=True开启字符串取值
    
    joblib.dump(news_features_dict, f"{output_dir}/news_feature_dict.pkl")
    print(f'新闻特征输出至 {output_dir}/news_feature_dict.pkl')

    del news_features_dict
    gc.collect()

    print("=" * 50 + " Processing embedding data " + "=" * 50)
    pca = PCA(n_components=emb_dim)

    df_text_emb = pl.read_parquet(text_emb_path).rename({'FacebookAI/xlm-roberta-base':'text_embedding'})
    df_image_emb = pl.read_parquet(image_emb_path)

    text_emb = pca.fit_transform(np.array(df_text_emb['text_embedding'].to_list()))
    image_emb = pca.fit_transform(np.array(df_image_emb['image_embedding'].to_list()))

    text_emb_dict = {id:emb for id, emb in zip(df_text_emb['article_id'], text_emb)}
    image_emb_dict = {id:emb for id, emb in zip(df_image_emb['article_id'], image_emb)}

    num_articles = len(article_ids_mapping_dict) + 1
    multimodal_matrix = np.zeros((num_articles, emb_dim*2), dtype=np.float32)


    for raw_id, mapped_id in article_ids_mapping_dict.items():
        t_emb = text_emb_dict.get(raw_id, np.zeros(emb_dim, dtype=np.float32))
        i_emb = image_emb_dict.get(raw_id, np.zeros(emb_dim, dtype=np.float32))
        multimodal_matrix[mapped_id] = np.concatenate([t_emb, i_emb])

    np.save(f'{output_dir}/multimadel_matrix.npy', multimodal_matrix)
    print(f'新闻内容embedding输出至 {output_dir}/multimadel_matrix.npy')

    del text_emb_dict, image_emb_dict, multimodal_matrix
    gc.collect()

    return article_ids_mapping_dict



def process_history_dynamic(
        history_path, 
        behavior_path, 
        article_ids_mapping_dict,
        neg_samples=14, 
    ):
    """
    处理曝光和用户点击行为，以及当此曝光时刻时用户的历史信息
    """
    
    print("=" * 50 + " Processing behavior data " + "=" * 50)
    # ==========================================
    # 1. 提取行为数据并构建画像映射字典
    # ==========================================
    df_behavior = pl.scan_parquet(behavior_path).drop_nulls('user_id')
    df_behavior = (
        df_behavior
        .drop_nulls('user_id')
        .select(['user_id','device_type','article_ids_inview','article_ids_clicked','gender','age'])
        .collect()
    )

    # 构建id映射
    user_ids_mapping_dict = create_id_mapping(df_behavior['user_id'])
    age_mapping_dict = create_id_mapping(df_behavior['age'])
    gender_mapping_dict = create_id_mapping(df_behavior['gender'])
    device_mapping_dict = create_id_mapping(df_behavior['device_type'])

    # ==========================================
    # 2. 曝光日志：负采样与 Point-wise 展开
    # ==========================================
    df_behavior = (
        df_behavior.lazy()  # 在进行复杂运算前重新开启惰性引擎
        .with_columns(
            # 将离散特征替换成id
            pl.col('user_id').replace(user_ids_mapping_dict, default=0),
            pl.col('age').replace(age_mapping_dict, default=0),
            pl.col('gender').replace(gender_mapping_dict, default=0),
            pl.col('device_type').replace(device_mapping_dict, default=0),
            clicked_times=pl.col('article_ids_clicked').list.len(),
        )
        .filter(pl.col('clicked_times') == 1)
        .with_columns(
            pl.col('article_ids_inview').list.set_difference(pl.col('article_ids_clicked'))
            .alias('neg_candidates')
        )
        .with_columns(
            pl.col('neg_candidates')
            .list.eval(pl.element().shuffle())
            .list.head(neg_samples)
            .alias('samples')
        )
        .with_columns(pl.concat_list(['samples', 'article_ids_clicked']))
        .drop(['neg_candidates','article_ids_inview','clicked_times'])
        .explode('samples')
        .with_columns(
            pl.col('samples').is_in(pl.col('article_ids_clicked')).cast(pl.Float32).alias('labels'),
            pl.col('samples').replace(article_ids_mapping_dict, default=0).alias('target_ids')
        )
        .drop(['samples', 'article_ids_clicked'])
        .collect()
    )
    gc.collect()

    # ==========================================
    # 3. 处理历史记录序列
    # ==========================================
    df_history = (
        pl.scan_parquet(history_path)
        .drop_nulls('user_id')
        .with_columns(
            pl.col('user_id').replace(user_ids_mapping_dict, default=0),
            pl.col('article_id_fixed').list.eval(pl.element().replace(article_ids_mapping_dict, default=0)).alias('history_ids')
        )
        .select(['user_id','history_ids'])
        .collect()
    )

    df_behavior = (
        df_behavior
        .join(df_history, on='user_id', how='left')
        .sample(fraction=1.0, shuffle=True)
    )

    behavior_dict = df_behavior.to_dicts()

    return behavior_dict, user_ids_mapping_dict, age_mapping_dict, gender_mapping_dict, device_mapping_dict


class EbnerdDataset(Dataset):
    def __init__(self, behavior_dict_list, news_features_dict, max_history_len=50):
        """
        :param behavior_dict_list: process_history_dynamic 输出的字典列表
        :param news_features_dict: 离线构建的文章属性字典 (包含 published_ts 等)
        """
        self.behaviors = behavior_dict_list
        self.news_features_dict = news_features_dict
        self.max_history_len = max_history_len

    def __len__(self):
        return len(self.behaviors)

In [2]:
import polars as pl
import numpy as np
import os
import shutil
from datetime import datetime
# from data_process import build_offline_article_vault, process_history_dynamic

article_path = "../data/articles.parquet"
behavior_path = "../data/behaviors.parquet"
history_path = "../data/history.parquet"
text_emb_path = "../data/roberta_vector.parquet"
image_emb_path = "../image_embedding.parquet"

def create_mini_official_dataset(source_dir, target_dir, n_rows=2000):
    """
    从官方数据集中抽取前 n_rows 行，构建用于测试的微型数据集
    """
    print(f"正在从 {source_dir} 切片前 {n_rows} 行数据...")
    os.makedirs(target_dir, exist_ok=True)
    
    # 1. 抽取基础表
    # 注意：请将这里的文件名替换为你真实存放的 parquet 文件名
    files_to_slice = [
        ("articles.parquet", "articles.parquet"),
        ("behaviors.parquet", "behaviors.parquet"),
        ("history.parquet", "history.parquet"),
        # 你的文本和图像 embedding 真实文件名填在这里
        ("roberta_vector.parquet", "text_emb.parquet"), 
        ("image_embeddings.parquet", "image_emb.parquet")
    ]
    
    for src_name, tgt_name in files_to_slice:
        src_path = os.path.join(source_dir, src_name)
        tgt_path = os.path.join(target_dir, tgt_name)
        
        if os.path.exists(src_path):
            # 极速读取前 n_rows 并保存
            df = pl.read_parquet(src_path).head(n_rows)
            df.write_parquet(tgt_path)
            print(f"成功切片并保存: {tgt_name} (共 {len(df)} 行)")
        else:
            print(f"找不到文件: {src_path}，请检查路径！")

if __name__ == "__main__":
    # ==========================================
    # 请在这里填入你官方数据集的真实路径！
    # ==========================================
    SOURCE_DIR = "../data" # 替换为你的真实路径
    TARGET_DIR = "./sliced_data"

    
    create_mini_official_dataset(SOURCE_DIR, TARGET_DIR, n_rows=2000)
    print("微型真实数据集构建完成！你可以用它来测试 data_process.py 了。")



正在从 ../data 切片前 2000 行数据...
成功切片并保存: articles.parquet (共 2000 行)
找不到文件: ../data/behaviors.parquet，请检查路径！
找不到文件: ../data/history.parquet，请检查路径！
成功切片并保存: text_emb.parquet (共 2000 行)
成功切片并保存: image_emb.parquet (共 2000 行)
微型真实数据集构建完成！你可以用它来测试 data_process.py 了。


In [3]:
def test_pipeline():
    tmp_dir = "./sliced_data"
    
    try:
        # ================= 测试离线物化 =================
        print("\n运行测试 1: 离线物品特征库构建")
        article_mapping = build_offline_article_vault(
            f"{tmp_dir}/articles.parquet", 
            f"{tmp_dir}/text_emb.parquet", 
            f"{tmp_dir}/image_emb.parquet", 
            tmp_dir, 
            emb_dim=2 # 测试时强制降为 2 维
        )
        
        # 检查多模态矩阵的形状
        matrix = np.load(f"{tmp_dir}/multimadel_matrix.npy")
        print("PCA 多模态矩阵形状:", matrix.shape) 
        assert matrix.shape == (len(article_mapping) + 1, 4), "矩阵维度应该等于 (文章数+1, 2*emb_dim)"
        
        # ================= 测试在线流水线 =================
        print("\n运行测试 2: 在线日志动态清洗与负采样")
        behavior_list, u_map, a_map, g_map, d_map = process_history_dynamic(
            f"{tmp_dir}/history.parquet", 
            f"{tmp_dir}/behaviors.parquet", 
            article_mapping,
            neg_samples=1 
        )
        
        print("\n终极输出的列表样本展示 (抽取第 1 条):")
        print(behavior_list[0])
        
        # 验证字典键名是否纯净
        expected_keys = {'user_id', 'device_type', 'gender', 'age', 'target_ids', 'labels', 'history_ids'}
        assert set(behavior_list[0].keys()) == expected_keys, "字典键名不符合预期！"
        
        print("\n数据管道全链路测试通过！")
        
    finally:
        print('over')
        # 阅后即焚，清理临时测试文件
        shutil.rmtree(tmp_dir)

if __name__ == "__main__":
    test_pipeline()


运行测试 1: 离线物品特征库构建
================================================== Processing article data ==================================================
over


FileNotFoundError: [Errno 2] No such file or directory: './sliced_data'

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DINAttention(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=32):
        """
        :param embedding_dim: 输入的embedding维度
        :param hidden_dim: 注意力网络的隐藏层维度
        """
        super(DINAttention, self).__init__()
        self.attention_mlp = nn.Sequential(
            nn.Linear(embedding_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, querys, keys, mask=None):
        """
        :param querys: [batch_size, 1, embedding_dim] 候选文章的embedding
        :param keys: [batch_size, seq_len, embedding_dim] 历史文章的embedding
        :param mask: [batch_size, seq_len] 用于屏蔽padding位置
        """
        # [!note] 使用.expand()将query的维度扩充成与keys相同以便于拼接，.expand(a,b,c)表示把三个维度分别扩充成a、b、c，-1表示不变
        input = torch.cat([
            querys.expand(-1, keys.shape[1], -1), 
            keys, 
            querys - keys, 
            querys * keys
        ], dim=-1)

        # attention_score形状(batch, seq_len, 1)
        attention_score = self.attention_mlp(input)

        # 不能写成if mask:
        if mask is not None:
            # mask形状是(batch, seq_len)，先将attention_score和mask的形状对齐
            attention_score = attention_score.masked_fill(mask.unsqueeze(-1)==0, -1e9)

        # attention_weights: (batch, seq_len)
        attention_weights = F.softmax(attention_score, dim=1)

        output = (attention_weights * keys).sum(dim=1)
        return output


In [35]:
batch_size = 2
seq_len = 5
embedding_dim = 64

dummy_query = torch.randn(batch_size, 1, embedding_dim)
dummy_keys = torch.randn(batch_size, seq_len, embedding_dim)

dummy_mask = torch.tensor([
    [1, 1, 1, 0, 0],
    [1, 0, 0, 0, 0]
])

attention_layer = DINAttention(embedding_dim=embedding_dim)
output = attention_layer(dummy_query, dummy_query, dummy_mask)

print(f"注意力层输出张量：{output}")
print(f"输出shape：{output.shape}")

注意力层输出张量：tensor([[-1.2862,  0.2428,  0.2764, -0.5735,  1.9728,  0.3346, -0.3415,  0.2661,
         -1.2710,  0.2144,  0.2606,  0.2255, -0.5664, -0.2140, -0.0885, -0.2160,
          0.4810, -1.7532,  0.1620,  0.8443, -1.1494, -0.1868,  0.4047, -0.7013,
         -0.6383, -1.0837,  0.8301, -0.4170,  2.0470, -0.1276, -0.7308,  0.3758,
          0.9434, -1.9515, -0.0676, -0.7067, -0.1819, -1.0793, -0.3996,  0.3067,
          0.9147, -0.5933,  0.4799,  0.4828, -0.7897, -0.1748,  0.4458, -0.4174,
          0.2267,  0.6886,  0.0504, -0.6373,  0.8735,  1.3754,  0.4776, -0.9871,
         -0.5047, -0.2634, -0.7050, -0.3504, -0.0924, -0.3496, -0.8801, -0.7616],
        [-0.9949, -1.6405,  0.7312, -0.5394, -0.8417, -0.9887, -0.7332,  0.8106,
          0.2032,  0.1971,  0.9209,  0.4625,  1.0218, -0.2244,  1.4687, -0.5412,
          0.2554, -0.8239,  0.5571, -1.2663,  1.8844,  0.2498, -0.3752, -0.8198,
          0.5122, -0.8520, -1.5577,  0.3654, -0.9831, -0.3435, -0.3100, -1.6945,
         -0.9473, 

In [31]:
a = torch.tensor([[[1,2,3]],[[4,5,6]]])
print(f"a: \n{a}")
print(a.shape)
b = torch.tensor([[[1,1,1], [1,1,1]],[[2,2,2], [2,2,2]]])
print(f"b: \n{b}")
print(b.shape)
print("===============")
c = a - b
d = a * b
print(f"c: \n{c}")
print(f"c.shape{c.shape}")
print(f"d: \n{d}")
print(f"d.shape{d.shape}")

concated = torch.cat([a.expand(-1, b.shape[1], -1), b, c, d], dim=-1)
print(f"concated: \n{concated}")
print(f"concated.shape: {concated.shape}")

a: 
tensor([[[1, 2, 3]],

        [[4, 5, 6]]])
torch.Size([2, 1, 3])
b: 
tensor([[[1, 1, 1],
         [1, 1, 1]],

        [[2, 2, 2],
         [2, 2, 2]]])
torch.Size([2, 2, 3])
c: 
tensor([[[0, 1, 2],
         [0, 1, 2]],

        [[2, 3, 4],
         [2, 3, 4]]])
c.shapetorch.Size([2, 2, 3])
d: 
tensor([[[ 1,  2,  3],
         [ 1,  2,  3]],

        [[ 8, 10, 12],
         [ 8, 10, 12]]])
d.shapetorch.Size([2, 2, 3])
concated: 
tensor([[[ 1,  2,  3,  1,  1,  1,  0,  1,  2,  1,  2,  3],
         [ 1,  2,  3,  1,  1,  1,  0,  1,  2,  1,  2,  3]],

        [[ 4,  5,  6,  2,  2,  2,  2,  3,  4,  8, 10, 12],
         [ 4,  5,  6,  2,  2,  2,  2,  3,  4,  8, 10, 12]]])
concated.shape: torch.Size([2, 2, 12])


In [1]:
import torch
import torch.nn as nn

class CrossLayer(nn.Module):
    def __init__(self, layer_num, embedding_dim):
        """
        :param layer_num: 交叉层的层数 (通常设为 2 到 3 层)
        :param input_dim: 输入特征的总维度 (比如时间和内容拼接后是 128 维)
        """
        super(CrossLayer, self).__init__()
        self.layer_num = layer_num
        self.cross_layers = nn.ModuleList([
            nn.Linear(embedding_dim, embedding_dim) for _ in range(layer_num)
            ])

    def forward(self, x0):
        """
        :param x0: 原始输入 [Batch, input_dim]
        """
        x_l = x0
        for i in range(self.layer_num):
            x_l = self.cross_layers[i](x_l) * x0 + x_l

        return x_l


class PopNet(nn.Module):
    """
    DCN架构
    """
    def __init__(self, recency_dim, content_dim, cross_layer_num=2, dnn_hidden_dims=[64, 32]):
        """
        经典并行 DCN 架构
        :param recency_dim: 时间特征维度
        :param content_dim: 内容特征维度
        :param cross_layer_num: 交叉网络的层数
        :param dnn_hidden_dims: 深度网络的隐藏层维度列表
        """
        super(PopNet, self).__init__()
        self.input_dim = recency_dim + content_dim
        self.cross_net = CrossLayer(layer_num=cross_layer_num, embedding_dim=self.input_dim)

        layers = []
        prev_dim = self.input_dim
        for dim in dnn_hidden_dims:
            layers.append(nn.Linear(prev_dim, dim))
            layers.append(nn.ReLU())
            prev_dim = dim

        self.mlp = nn.Sequential(*layers)

        concat_dim = self.input_dim + dnn_hidden_dims[-1]
        self.final_linear = nn.Linear(concat_dim, 1)

    def forward(self, recency_emb, content_emb):
        assert recency_emb.dim() == content_emb.dim(), "recency_emb和content_emb维度不同"
        x0 = torch.cat([recency_emb, content_emb], dim=-1)

        cross_out = self.cross_net(x0)
        mlp_out = self.mlp(x0)

        final_input = torch.cat([cross_out, mlp_out], dim=-1)
        
        return self.final_linear(final_input)

        

In [3]:
recency_dim = 32
content_dim = 64
batch_size = 2

dummy_recency = torch.randn(batch_size, recency_dim)
dummy_content = torch.randn(batch_size, content_dim)
pop_net = PopNet(recency_dim=recency_dim, content_dim=content_dim)

pop_out = pop_net(dummy_recency, dummy_content)

print(pop_out)
print(pop_out.shape)




tensor([[ 0.9907],
        [-0.7686]], grad_fn=<AddmmBackward0>)
torch.Size([2, 1])


In [ ]:
import torch
import torch.nn as nn

class Gate(nn.Module):
    def __init__(self, user_dim, recency_dim, content_dim, dnn_hidden_dims=[64,32]):
        super(Gate, self).__init__()
        self.input_dim = user_dim + recency_dim + content_dim

        mlp_list = []
        last_dim = self.input_dim
        for dim in dnn_hidden_dims:
            mlp_list.append(nn.Linear(last_dim, dim))
            mlp_list.append(nn.ReLU())
            last_dim = dim

        self.mlp = nn.Sequential(*mlp_list, nn.Linear(dnn_hidden_dims[-1], 1), nn.Sigmoid())

    def forward(self, user_emb, recency_emb, content_emb):
        assert user_emb.dim() == recency_emb.dim() == content_emb.dim(), "user_dim, recency_dim, content_dim维度不一致"

        concat_tensor = torch.cat([user_emb, recency_emb, content_emb], dim=-1)
        
        return self.mlp(concat_tensor)

        

In [63]:
batch_size = 2
user_dim = 64
recency_dim = 32
content_dim = 64

dummmy_user_emb = torch.randn(batch_size, user_dim)
dummmy_recency_emb = torch.randn(batch_size, recency_dim)
dummmy_content_emb = torch.randn(batch_size, content_dim)

gate = Gate(user_dim, recency_dim, content_dim)
gate_output = gate(dummmy_user_emb, dummmy_recency_emb, dummmy_content_emb)

print(gate_output)
print(gate_output.shape)

TypeError: 'Tensor' object is not callable